# DurakZero Live Helper

Use this notebook to attach the trained DurakZero model to a live online game. It consumes decoded packet strings (either via `pyshark` or from a log file) and streams model recommendations in real time.

## Prerequisites

* Ensure `tshark` is available on your system if you plan to capture packets with `pyshark`.
* The helper expects the JSON-like text lines captured in your existing notebook (see the example in the prompt).
* When sourcing from a log file, keep appending new packet lines to that file (e.g. via the capture notebook).

In [ ]:
# Optional: install capture dependencies in this environment
%pip install --quiet pyshark nest_asyncio

In [ ]:
# Configuration
CHECKPOINT_PATH = 'model.tar'  # update to your checkpoint location
DEVICE = 'cpu'                # 'cpu', 'cuda', 'cuda:0', '0', etc.
PLAYER_ID = None              # Optionally force 0 or 1 if auto-detection fails
SOURCE = 'pyshark'            # 'pyshark' or 'file'
INTERFACE = 'waydroid0'       # network interface for pyshark
IP_FILTER = 'ip.addr == 65.21.92.166'  # display filter (set to None to disable)
LOG_FILE = '/path/to/durak_packets.txt'  # used only when SOURCE == 'file'

In [ ]:
import asyncio
from pathlib import Path

import torch
import nest_asyncio

from douzero.evaluation.simulation import load_model
from douzero.live import LiveDurakTracker
from tools.durak_live_helper import _capture_pyshark, _tail_file, _resolve_device

nest_asyncio.apply()

device = _resolve_device(DEVICE)
model = load_model(CHECKPOINT_PATH, device=DEVICE)
tracker = LiveDurakTracker(model=model, device=device, player_id=PLAYER_ID, verbose=True)
print(f'Tracker initialised on {device}.')

In [ ]:
async def stream_packets():
    if SOURCE == 'file':
        if not LOG_FILE:
            raise ValueError('LOG_FILE must be set when SOURCE == "file".')
        iterator = _tail_file(Path(LOG_FILE))
        print(f'Tailing {LOG_FILE}... Press Stop to end.')
    else:
        iterator = _capture_pyshark(INTERFACE, IP_FILTER)
        print(f'Listening on {INTERFACE} (filter: {IP_FILTER or "none"})... Press Stop to end.')
    for line in iterator:
        tracker.process_raw_line(line)
        await asyncio.sleep(0)

# Run the capture loop. Interrupt or Stop the cell to quit.
await stream_packets()